# Vie-GameEmo — Training (Simplified)

Notebook này gọi trực tiếp các scripts của project thay vì inline code.

**Pipeline:**
```
import_labels.py → stage0_preprocess.py → transcribe.py → extract_features.py → train.py
```

**Dataset layout cần có trước:**
```
data/
├── raw_videos/train/  ← clips .mp4
├── raw_videos/val/
├── raw_videos/test/
├── labels/train.json  ← [{id, video, choice, confidence}]
├── labels/val.json
└── labels/test.json
```


In [ ]:
# ============================================================
# CELL 1 — Môi trường
# ============================================================
import os, sys
WORKING = os.getcwd()
print(f'Working dir: {WORKING}')

# GPU check
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB)')
else:
    print('⚠️  No GPU — training will be very slow')


In [ ]:
# ============================================================
# CELL 2 — CẤU HÌNH
# ============================================================

# --- Paths ---
DATASET_INPUT = '/kaggle/input/vie-gameemo-dataset'      # Kaggle input dataset
DATASET_LOCAL = os.path.join(WORKING, 'data')             # local fallback
PROJECT_INPUT = '/kaggle/input/vie-gameemo-code'          # project code

# --- Training ---
EPOCHS       = 30
BATCH_SIZE   = 16
FUSION_TYPE  = 'conv_attention_4m'
MIXED_PREC   = 'bf16'

# --- Optional stages ---
TRAIN_COGNITION = False
TRAIN_RLVR      = False


In [ ]:
# ============================================================
# CELL 3 — Cài thư viện
# ============================================================
!pip install -q \
    transformers>=4.45.0 \
    peft>=0.12.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.33.0 \
    faster-whisper>=1.0.3 \
    fasttext-wheel \
    scikit-learn \
    pydantic>=2.0 \
    librosa \
    torchvision \
    torchaudio


In [ ]:
# ============================================================
# CELL 4 — Setup project
# ============================================================
import shutil, subprocess

PROJECT_DIR = os.path.join(WORKING, 'vie-gameemo-skeleton')

if os.path.exists(PROJECT_INPUT):
    if not os.path.exists(PROJECT_DIR):
        shutil.copytree(PROJECT_INPUT, PROJECT_DIR)
    print(f'Project: Kaggle input → {PROJECT_DIR}')
elif not os.path.exists(PROJECT_DIR):
    GITHUB_URL = 'https://github.com/rhy221/vie-gameemo-skeleton.git'
    subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, PROJECT_DIR], check=True)

# Add to path
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Detect dataset
if os.path.exists(DATASET_INPUT):
    DATA_DIR = DATASET_INPUT
else:
    DATA_DIR = DATASET_LOCAL

SCRIPTS = os.path.join(PROJECT_DIR, 'scripts')
CONFIG  = os.path.join(PROJECT_DIR, 'config.yaml')
print(f'Data:    {DATA_DIR}')
print(f'Scripts: {SCRIPTS}')


## Bước 1 — Import labels + Tiền xử lý


In [ ]:
# ============================================================
# CELL 5 — Import labels → annotations + splits.json
# ============================================================
!python {SCRIPTS}/import_labels.py --data-root {DATA_DIR}


In [ ]:
# ============================================================
# CELL 6 — Tiền xử lý: tách audio + frames từ raw videos
# ============================================================
!python {SCRIPTS}/stage0_preprocess.py \
    --config {CONFIG} \
    --videos-dir {DATA_DIR}/raw_videos \
    --skip-webcam-detect


In [ ]:
# ============================================================
# CELL 7 — ASR: transcribe audio → cập nhật annotations
# ============================================================
!python {SCRIPTS}/transcribe.py --config {CONFIG}


## Bước 2 — Trích xuất features


In [ ]:
# ============================================================
# CELL 8 — Extract + cache features (4 modality encoders)
# ============================================================
!python {SCRIPTS}/extract_features.py --config {CONFIG}


## Bước 3 — Training


In [ ]:
# ============================================================
# CELL 9 — Stage 1: Perception Training
# ============================================================
!python {SCRIPTS}/train.py \
    --config {CONFIG} \
    --stage perception \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --fusion {FUSION_TYPE}


In [ ]:
# ============================================================
# CELL 10 — Eval trên test split
# ============================================================
!python {SCRIPTS}/eval.py --config {CONFIG}


In [ ]:
# ============================================================
# CELL 10b — Phân tích kết quả + gợi ý tinh chỉnh
# ============================================================
# Đọc eval.json → phân tích per-class, confusion, rare class, language gap
# → in ra recommendations cụ thể.
!python {SCRIPTS}/analyze.py \
    --config {CONFIG} \
    --eval-json outputs/results/eval.json \
    --with-fragmentation


## Bước 4 — (Tùy chọn) Cognition + RLVR


In [ ]:
# ============================================================
# CELL 11 — Stage 2: Cognition (joint LLM + adapter)
# ============================================================
if TRAIN_COGNITION:
    !python {SCRIPTS}/train.py \
        --config {CONFIG} \
        --stage cognition \
        --resume-from outputs/checkpoints/perception_best.pt
else:
    print('TRAIN_COGNITION=False — bỏ qua')


In [ ]:
# ============================================================
# CELL 12 — RLVR (LLM-4, tùy chọn)
# ============================================================
if TRAIN_RLVR:
    !python {SCRIPTS}/train_rlvr.py \
        --config {CONFIG} \
        --phase cold-start \
        --base-model Qwen/Qwen2.5-1.5B-Instruct \
        --epochs 2
else:
    print('TRAIN_RLVR=False — bỏ qua')


In [ ]:
# ============================================================
# CELL 13 — Lưu checkpoint để download
# ============================================================
import zipfile
from pathlib import Path

CKPT_DIR = 'outputs/checkpoints'
ckpt_files = list(Path(CKPT_DIR).glob('*.pt'))
print(f'Checkpoints ({len(ckpt_files)}):')
for f in ckpt_files:
    print(f'  {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

archive = os.path.join(WORKING, 'vie_gameemo_checkpoints.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for ckpt in ckpt_files:
        zf.write(str(ckpt), f'checkpoints/{ckpt.name}')
    zf.write(CONFIG, 'config.yaml')

print(f'\n✅ Archive: {archive} ({os.path.getsize(archive)/1e6:.1f} MB)')
